<br>
<p float="right">
  <img src=attachment:nanoHUB_logo_color.png width="25%" height='10%' align="right" /> 
</p>

# Demonstration: LAMMPS Parser on LLM-Automatic Script Generation

### <i>Ethan Holbrook, Juan C. Verduzco, </i>  and <i>Alejandro Strachan </i>
### Materials Engineering, Purdue University <br>

## Overview

This notebook is the third stage of the evaluation pipeline. It executes parsed scripts with LAMMPS, records runtime outcomes, and identifies scripts that fail during simulation setup or execution.

To make evaluation practical, the notebook uses shortened run lengths for rapid testing while preserving the overall script structure. It also includes an auxiliary pair-style neutralization step used to separate interaction-model failures from other script-construction errors.

You will need to have a LAMMPS executable file to point to in your run environment. Specified at the top. 

## Tips
1. Found a bug? Email holbrooe@purdue.edu

<br><br><br>

# Libraries

In [53]:
import os
import json
import sys
import shutil
import re

from dotenv import load_dotenv
load_dotenv()

from lark import tree
from lammps_ast.sanitizer import sanitize
from lammps_ast.parser import parse_to_AST

import lammps_ast
print(dir(lammps_ast))
print(lammps_ast.__path__)

import openai
from openai import OpenAI

import numpy as np
import pandas as pd

import subprocess

from colorama import Fore, Style

from importlib.metadata import version
print(version("lammps-ast"))

# import anthropic

import ast # python ast

LAMMPS_EXECUTABLE = "/apps/spack/negishi/apps/lammps/20220623-gcc-12.2.0-eayjiz7/bin/lmp"

['__author__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'error_handler', 'grammar', 'parse_to_AST', 'parser', 'sanitize', 'sanitizer', 'transformer']
['/home/holbrooe/.conda/envs/2022.10-py39/gst/lib/python3.12/site-packages/lammps_ast']
0.1.7


In [56]:
# read parser results
new_df = pd.read_pickle('parsing_df.pkl')
# display(new_df)

## Initialization of directory variables

In [57]:
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir))
scripts_dir = os.path.join(parent_dir,"generated_scripts")
scripts_base = os.path.join(parent_dir,"generated_scripts")

print(scripts_dir)

prompt_dirs = ['prompt1','prompt2','prompt3']
model_dirs = ['gpt-4o','gpt-4.1','gpt-o3','claude-opus-4','gpt-5'] #,'gpt-4o-search',]
# model_dirs = [] #,'gpt-4o-search',]
model_names = ['gpt-4o-2024-08-06','gpt-4.1-2025-04-14','o3-2025-04-16','claude-4-opus-20250514','gpt-5-2025-08-07']
# model_names = ['gpt-5-2025-08-07']

/home/holbrooe/LAMMPS-AST/EvaluationPipelineExample/generated_scripts


In [58]:
prompt_choice = 1
model_choice = 0

prompt_type = prompt_dirs[prompt_choice-1]
print(prompt_type)
model_type = model_dirs[model_choice]
model_name = model_names[model_choice]
print(model_type,model_name)

trial=0
# output_filename = os.path.join(prompt_type, model_type,f'P_{prompt_type}-M_{model_type}-T_{trial}.in')

print(prompt_dirs)
print(model_dirs)
print(model_names)

# prompt_dict = {'prompt1':prompts[0],'prompt2':prompts[1],'prompt3':prompts[2]}

model_dict = {model_dirs[0]:model_names[0],
              model_dirs[1]:model_names[1],
              model_dirs[2]:model_names[2],
              model_dirs[3]:model_names[3],
              model_dirs[4]:model_names[4],
             }

prompt1
gpt-4o gpt-4o-2024-08-06
['prompt1', 'prompt2', 'prompt3']
['gpt-4o', 'gpt-4.1', 'gpt-o3', 'claude-opus-4', 'gpt-5']
['gpt-4o-2024-08-06', 'gpt-4.1-2025-04-14', 'o3-2025-04-16', 'claude-4-opus-20250514', 'gpt-5-2025-08-07']


In [59]:
#what we see
print(scripts_dir)
prompt_numbers = sorted(next(os.walk(scripts_dir))[1])
print(prompt_numbers)

prompt_model_map1 = {}

for prompt in prompt_numbers:
    prompt_dir = os.path.join(scripts_dir, prompt)
    models = sorted(next(os.walk(prompt_dir))[1])  
    prompt_model_map1[prompt] = models

# Display detected structure
for prompt, models in prompt_model_map1.items():
    print(f"📂 {prompt}: {', '.join(models)}")

/home/holbrooe/LAMMPS-AST/EvaluationPipelineExample/generated_scripts
['prompt1', 'prompt2', 'prompt3']
📂 prompt1: gpt-4o
📂 prompt2: gpt-4o
📂 prompt3: gpt-4o


# Run?
Got to first shorten run lines to 10, to make it quicker

In [60]:
def modify_run_lines(file_path, output_path):
    with open(file_path, 'r') as file:
        lines = file.readlines()
    
    modified_lines = []
    
    for line in lines:
        # Strip whitespace and check if the line is not empty
        if line.strip():
            words = line.split()
            if words[0] == 'run' and len(words) > 1:
                words[1] = '10'  # Replace the second entry
                line = ' '.join(words) + '\n'  # Reconstruct the line
        modified_lines.append(line)  # Append modified or original line
    
    # Write to a new file
    with open(output_path, 'w') as output_file:
        output_file.writelines(modified_lines)


def modify_run_lines(file_path, output_path, potential_dir, 
                     potential_needed=True,prompt='prompt1'):
    '''ensures'''
    '''added an additional provision for changing file names for potentials'''
    
    with open(file_path, 'r') as file:
        lines = file.readlines()
    
    modified_lines = []
    
    for line in lines:
        stripped_line = line.strip()
        
        # Modify 'run' commands
        if stripped_line.startswith('run'):
            words = line.split()
            if len(words) > 1:
                words[1] = '10'  # Replace the second argument
                line = ' '.join(words) + '\n'
        
        if potential_needed == True:
            # Modify 'pair_coeff' lines
            if stripped_line.startswith('pair_coeff'):
                words = line.split()
                # print(words)
                if len(words) > 2:  # Ensure it's a valid line
                    potential_file = words[3]  # Extract the potential filename
                    if not potential_file.startswith('/'):  # Only modify if it's not an absolute path
                        print('yes')
#                         words[3] = os.path.join(potential_dir, os.path.basename(potential_file))
                        words[3] = os.path.join(potential_dir, f'{prompt}.potential')
                        print(words)
                        
                        line = ' '.join(words) + '\n'
        
        modified_lines.append(line)

    # Write to the output file
    with open(output_path, 'w') as output_file:
        output_file.writelines(modified_lines)

In [61]:
print(scripts_dir)

/home/holbrooe/LAMMPS-AST/EvaluationPipelineExample/generated_scripts


In [62]:

# prompt_model_map2 = {'prompt3':['gpt-4.1']}


In [63]:
# scripts_dir = '/home/nanohub/ethan2/tool-dev/lammpsparser/generated_scripts'
short_run_scripts_dir = 'short_run_scripts'
os.makedirs(short_run_scripts_dir, exist_ok=True)  # Ensure directories exist

potential_dir = 'potentials'

# prompts = ['P2','P3','strachan','zapiain']
# models = ['4o','o1','o3-mini-high'] #,'4_5','o1-extra']
def modify_for_short_runs(prompt_model_map,input_script_dir, output_script_dir, potential_dir):
    for prompt, models in prompt_model_map.items():
        prompt_dir = os.path.join(output_script_dir,prompt)
        os.makedirs(prompt_dir, exist_ok=True)
        for model in models:
            # print(prompt,model)
            model_dir = os.path.join(output_script_dir,prompt,model)
            print(model_dir)
            os.makedirs(model_dir, exist_ok=True)
#             scripts = os.listdir(model_dir)
            # scripts = [f'{prompt}-{model}-T{trial}.in' for trial in range(10)]
            scripts = [f'{prompt}-{model}-T{trial}.in' for trial in range(1)]
            print(scripts)

            for script in scripts:
                input_fp = os.path.join(input_script_dir,prompt,model,script)

                output_fp = os.path.join(output_script_dir,prompt,model,script)
                if prompt == 'prompt1':
                    modify_run_lines(input_fp, output_fp, potential_dir, prompt='prompt1')
                elif prompt== 'prompt2':
                    modify_run_lines(input_fp, output_fp, potential_dir, prompt='prompt2')
                else:
                    modify_run_lines(input_fp, output_fp, potential_dir, prompt='prompt3')
                
modify_for_short_runs(prompt_model_map1,scripts_dir,short_run_scripts_dir,potential_dir)                
# modify_for_short_runs(prompt_model_map1,sanitized_scripts_dir,short_run_scripts_dir,potential_dir) ### cant do this because the sanitizer is lenient on variable expressions. 
# Eg. if spaces are left in between a variable expression, the sanitizer will handle it but LAMMPS would see it as separate arguments.
# short_runs(prompt_model_map2,scripts_dir,short_run_scripts_dir,potential_dir)



short_run_scripts/prompt1/gpt-4o
['prompt1-gpt-4o-T0.in']
yes
['pair_coeff', '*', '*', 'potentials/prompt1.potential', 'Al']
short_run_scripts/prompt2/gpt-4o
['prompt2-gpt-4o-T0.in']
yes
['pair_coeff', '*', '*', 'potentials/prompt2.potential', 'Ni']
short_run_scripts/prompt3/gpt-4o
['prompt3-gpt-4o-T0.in']
yes
['pair_coeff', '*', '*', 'potentials/prompt3.potential', 'Nb', 'Nb']


In [64]:
def run_lammps(lmp_exec, input_file, log_file):
    """
    Launch LAMMPS and write its log. On failure, raise a RuntimeError
    whose message is the last non-blank line of stderr (or stdout).
    """
    env = os.environ.copy()
    lammps_dir = "/apps/share64/debian10/lammps/lammps-02Aug23"
    env["LD_LIBRARY_PATH"] = os.path.join(lammps_dir, "lib") \
                             + ":" + env.get("LD_LIBRARY_PATH", "")

    cmd = [lmp_exec, "-in", input_file, "-l", log_file]
    result = subprocess.run(cmd, env=env, capture_output=True, text=True)

    # If LAMMPS itself printed to stdout/stderr, you can log it too:
    print("LAMMPS STDOUT:\n", result.stdout)
    print("LAMMPS STDERR:\n", result.stderr)

    if result.returncode != 0:
        # pick last non-empty stderr line, or fallback to stdout
        lines = [L for L in result.stderr.splitlines() if L.strip()]
        if not lines:
            lines = [L for L in result.stdout.splitlines() if L.strip()]
        err_msg = lines[-2:] if lines else f"exit code {result.returncode}"
        raise RuntimeError(err_msg)

    # otherwise no return value needed; no exception means "success"

In [82]:
def do_runs(prompt_model_map, df, scripts_dir, logs_dir, LAMMPS_EXECUTABLE):
    """
    Given your parse‐results df (with ['prompt','model','trial',…]),
    fill in df['run'] with either "Success", "not parsed", or the error message from LAMMPS.
    """
    df = df.copy()
    df['run'] = ''  # will store either "Success", "not parsed", or the caught error

    for prompt, models in prompt_model_map.items():
        for model in models:
            src_dir = os.path.join(scripts_dir, prompt, model)
            out_dir = os.path.join(logs_dir,  prompt, model)
            os.makedirs(out_dir, exist_ok=True)

            for trial in range(1):
            # for trial in range(10):
                # Build mask first to select matching row(s)
                mask = (
                    (df['prompt'] == prompt) &
                    (df['model']  == model)  &
                    (df['trial']  == trial)
                )

                # Check if parsed failed
                if (df.loc[mask, 'parsed'] != True).any():
                    df.loc[mask, 'run'] = 'not parsed'
                    continue

                script_name = f"{prompt}-{model}-T{trial}.in"
                input_path  = os.path.join(src_dir, script_name)
                name        = f"lmmp-{prompt}-{model}-{trial}"
                log_path    = os.path.join(out_dir, name + ".log")

                # run LAMMPS & catch errors
                try:
                    run_lammps(
                        lmp_exec   = LAMMPS_EXECUTABLE,
                        input_file = input_path,
                        log_file   = log_path
                    )
                    status = True
                except Exception as e:
                    status = str(e)  # the error message we raised
                    print('-----------------------------------------')
                    print(status)
                    print('-----------------------------------------')
                df.loc[mask, 'run'] = status

    return df

logs_dir = 'logs2'
run_df = do_runs(prompt_model_map1,new_df,short_run_scripts_dir,logs_dir,LAMMPS_EXECUTABLE)
# prompt_model_map3 = {'prompt1': ['claude-4-opus']}
# run_df = do_runs(prompt_model_map3,new_df,short_run_scripts_dir,logs_dir)


LAMMPS STDOUT:
 ERROR on proc 0: Cannot open input script short_run_scripts/prompt1/gpt-4o/prompt1-gpt-4o-T0.in: No such file or directory (src/src/lammps.cpp:518)
Last command: (unknown)

LAMMPS STDERR:
 --------------------------------------------------------------------------
MPI_ABORT was invoked on rank 0 in communicator MPI_COMM_WORLD
with errorcode 1.

NOTE: invoking MPI_ABORT causes Open MPI to kill all MPI processes.
You may or may not see output from other processes, depending on
exactly when Open MPI kills them.
--------------------------------------------------------------------------

-----------------------------------------
['exactly when Open MPI kills them.', '--------------------------------------------------------------------------']
-----------------------------------------
LAMMPS STDOUT:
 ERROR on proc 0: Cannot open input script short_run_scripts/prompt2/gpt-4o/prompt2-gpt-4o-T0.in: No such file or directory (src/src/lammps.cpp:518)
Last command: (unknown)

LAMMPS

In [66]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(run_df)

,prompt,model,trial,sanitized,parsed,ast_path,run
0,prompt1,gpt-4o,0,True,True,asts/prompt1/gpt-4o/prompt1-gpt-4o-T0.ast.pkl,True
1,prompt2,gpt-4o,0,True,True,asts/prompt2/gpt-4o/prompt2-gpt-4o-T0.ast.pkl,['ERROR: Incorrect args for pair coefficients ...
2,prompt3,gpt-4o,0,True,True,asts/prompt3/gpt-4o/prompt3-gpt-4o-T0.ast.pkl,['ERROR: Incorrect args for pair coefficients ...


In [78]:
def modify_pair_style(file_path, output_path):
    with open(file_path, 'r') as file:
        lines = file.readlines()
    
    modified_lines = []
    
    for line in lines:
        stripped_line = line.strip()
        
        # Modify 'pair_coeff' lines
        if stripped_line.startswith('pair'):
            if stripped_line.startswith('pair_style'):
                line = '\n'
            else:
                line = 'pair_style zero 10.0\n'+'pair_coeff * *\n'+'mass 1 27\n'
                
        modified_lines.append(line)

    # Write to the output file
    with open(output_path, 'w') as output_file:
        output_file.writelines(modified_lines)
        
def pair_style_change(prompt_model_map, df, scripts_dir, pair_dir, log_dir):
    """

    """
    df = df.copy()
#     df['pair_style'] = ''  # will store either "Success", "not parsed", or the caught error

    for prompt, models in prompt_model_map.items():
        for model in models:
            src_dir = os.path.join(scripts_dir, prompt, model)
            out_dir = os.path.join(pair_dir, prompt, model)
            log_out_dir = os.path.join(log_dir, prompt, model)
            os.makedirs(out_dir, exist_ok=True)
            os.makedirs(log_out_dir, exist_ok=True)

            # for trial in range(10):
            for trial in range(1):
                # Build mask first to select matching row(s)
                mask = (
                    (df['prompt'] == prompt) &
                    (df['model']  == model)  &
                    (df['trial']  == trial)
                )
                print(mask)
                # Check if failed by pair_style
                if df.loc[mask, 'run'].iloc[0] != True and df.loc[mask, 'run'].iloc[0] != 'not parsed':
#                     print(df.loc[mask, 'run'].iloc[0])
#                     print('-------------')
                    print(ast.literal_eval(df.loc[mask, 'run'].iloc[0])[1])
                    
                    if ast.literal_eval(df.loc[mask, 'run'].iloc[0])[1].startswith('Last command: pair'):
                        script_name = f"{prompt}-{model}-T{trial}.in"
                        input_path  = os.path.join(src_dir, script_name)
                        out_path    = os.path.join(out_dir, script_name)
                        name        = f"lmmp-{prompt}-{model}-{trial}"
                        log_path    = os.path.join(log_out_dir, name + ".log")
                        modify_pair_style(input_path,out_path)
                    # run LAMMPS & catch errors
                        try:
                            run_lammps(
                                lmp_exec   = "/apps/share64/debian10/lammps/lammps-02Aug23/bin/lmp_serial",
                                input_file = out_path,
                                log_file   = log_path
                            )
                            status = True
                        except Exception as e:
                            status = str(e)  # the error message we raised
                            print(e)
                            
                        df.loc[mask, 'pair_run'] = status
                    else:
                        df.loc[mask, 'pair_run'] = False

                else:
                    df.loc[mask, 'pair_run'] = 'n/a'

    return df

In [79]:
print(prompt_model_map1)
display(run_df)
short_run_scripts_dir
pair_dir
pair_log_dir

{'prompt1': ['gpt-4o'], 'prompt2': ['gpt-4o'], 'prompt3': ['gpt-4o']}


,prompt,model,trial,sanitized,parsed,ast_path,run
0,prompt1,gpt-4o,0,True,True,asts/prompt1/gpt-4o/prompt1-gpt-4o-T0.ast.pkl,True
1,prompt2,gpt-4o,0,True,True,asts/prompt2/gpt-4o/prompt2-gpt-4o-T0.ast.pkl,['ERROR: Incorrect args for pair coefficients ...
2,prompt3,gpt-4o,0,True,True,asts/prompt3/gpt-4o/prompt3-gpt-4o-T0.ast.pkl,['ERROR: Incorrect args for pair coefficients ...


'pair_change_logs2'

In [80]:
pair_dir = 'pair_change2'
os.makedirs(pair_dir,exist_ok=True)
pair_log_dir = 'pair_change_logs2'
os.makedirs(pair_log_dir,exist_ok=True)
# pair_df = pair_style_change(prompt_model_map3, run_df, short_run_scripts_dir, pair_dir, pair_log_dir)
pair_df = pair_style_change(prompt_model_map1, run_df, short_run_scripts_dir, pair_dir, pair_log_dir)

0     True
1    False
2    False
dtype: bool
0    False
1     True
2    False
dtype: bool
Last command: pair_coeff * * potentials/prompt2.potential Ni
[Errno 2] No such file or directory: '/apps/share64/debian10/lammps/lammps-02Aug23/bin/lmp_serial'
0    False
1    False
2     True
dtype: bool
Last command: pair_coeff * * potentials/prompt3.potential Nb Nb
[Errno 2] No such file or directory: '/apps/share64/debian10/lammps/lammps-02Aug23/bin/lmp_serial'


In [81]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(pair_df)

pair_df.to_pickle('final_pair_df.pkl')

,prompt,model,trial,sanitized,parsed,ast_path,run,pair_run
0,prompt1,gpt-4o,0,True,True,asts/prompt1/gpt-4o/prompt1-gpt-4o-T0.ast.pkl,True,n/a
1,prompt2,gpt-4o,0,True,True,asts/prompt2/gpt-4o/prompt2-gpt-4o-T0.ast.pkl,['ERROR: Incorrect args for pair coefficients ...,[Errno 2] No such file or directory: '/apps/sh...
2,prompt3,gpt-4o,0,True,True,asts/prompt3/gpt-4o/prompt3-gpt-4o-T0.ast.pkl,['ERROR: Incorrect args for pair coefficients ...,[Errno 2] No such file or directory: '/apps/sh...
